In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
State_crop_yeild=pd.read_csv(r"D:\researrch\agree.culture.Ai\code\dataset\state_wise_crop_yild.csv")
State_crop_yeild.head()

,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
0,Arecanut,1997,Whole Year,Assam,73814.0,56708,2051.4,7024878.38,22882.34,0.796087
1,Arhar/Tur,1997,Kharif,Assam,6637.0,4685,2051.4,631643.29,2057.47,0.710435
2,Castor seed,1997,Kharif,Assam,796.0,22,2051.4,75755.32,246.76,0.238333
3,Coconut,1997,Whole Year,Assam,19656.0,126905000,2051.4,1870661.52,6093.36,5238.051739
4,Cotton(lint),1997,Kharif,Assam,1739.0,794,2051.4,165500.63,539.09,0.420909


In [4]:
# state wise rainfal vs crop prediction model
from sklearn.model_selection import train_test_split
# from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score



In [5]:
X=State_crop_yeild.drop('Yield',axis=1)
y=State_crop_yeild['Yield']

# Identify categorical and numerical features
categorical_features = ['Crop', 'Season', 'State']
numerical_features = [  'Production', 'Annual_Rainfall','Area']

# Create a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numerical_features)
    ])

# Import RandomForestRegressor (since Yield is continuous, it's a regression problem)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Create the model pipeline
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('regressor', RandomForestRegressor(random_state=42))])

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Train the model
print("Training the model...")
model_pipeline.fit(X_train, y_train)
print("Model training complete.")

# Make predictions
y_pred = model_pipeline.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nModel Evaluation:")
print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared: {r2:.2f}")

# Display a few predictions vs actual values
results_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
print("\nFirst 10 Actual vs Predicted Yields:")
print(results_df.head(10))

print("saving the model")

import joblib
joblib.dump(model_pipeline, 'crop_prediction_model.pkl')



joblib.dump(label_encoder, 'crop_label_encoder.pkl')

Training the model...
Model training complete.

Model Evaluation:
Mean Squared Error: 313261.22
R-squared: 0.98

First 10 Actual vs Predicted Yields:
   Actual  Predicted
0   10259   10438.39
1    6949    6361.78
2   11051   11062.32
3    4297    4360.17
4    2260    2672.74
5    1341     982.90
6    3903    3954.20
7    8153    8162.47
8   13243   13237.95
9    7998    8474.31
saving the model


['crop_label_encoder.pkl']

In [ ]:
# Initialize dictionaries to store models and evaluation results for each state
state_models = {}
state_metrics = {}

# Get unique states from the dataset
unique_states = State_crop_yeild['State'].unique()

print(f"Starting to train models for {len(unique_states)} unique states...")

for state in unique_states:
    print(f"\n--- Training model for {state} ---")
    
    # Filter data for the current state
    state_df = State_crop_yeild[State_crop_yeild['State'] == state].copy()

    # Skip states with insufficient data for a meaningful train/test split
    # A minimum of 5 samples is set as a heuristic to allow for at least 1 test sample and 4 training samples.
    if len(state_df) < 5:
        print(f"  Skipping {state}: Insufficient data ({len(state_df)} samples). Requires at least 5 samples.")
        continue

    # Define features (X) and target (y) for the current state
    # The 'State' column is constant for the filtered DataFrame, so it's dropped from features.
    X_state = state_df.drop(['Yield', 'State'], axis=1)
    y_state = state_df['Yield']

    # Identify categorical and numerical features for this state (excluding 'State')
    categorical_features_state = ['Crop', 'Season']
    numerical_features_state = ['Crop_Year', 'Area', 'Production', 'Annual_Rainfall', 'Fertilizer', 'Pesticide']

    # Ensure only existing columns are passed to the ColumnTransformer
    current_categorical_features = [f for f in categorical_features_state if f in X_state.columns]
    current_numerical_features = [f for f in numerical_features_state if f in X_state.columns]

    # Check if there are any features left to train on
    if not current_categorical_features and not current_numerical_features:
        print(f"  Skipping {state}: No valid features found after filtering relevant columns.")
        continue

    # Create a column transformer specific to this state's data
    # OneHotEncoder handles unseen categories gracefully with handle_unknown='ignore'
    state_preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), current_categorical_features),
            ('num', 'passthrough', current_numerical_features)
        ],
        remainder='drop' # Drop any columns not explicitly specified
    )

    # Create the model pipeline for the current state
    state_model_pipeline = Pipeline(steps=[
        ('preprocessor', state_preprocessor),
        ('regressor', RandomForestRegressor(random_state=42))
    ])

    # Split the data for the current state into training and testing sets
    try:
        X_train_state, X_test_state, y_train_state, y_test_state = train_test_split(
            X_state, y_state, test_size=0.2, random_state=42
        )
    except ValueError as e:
        print(f"  Skipping {state}: Could not split data ({e}). This usually happens with too few samples or target values.")
        continue

    # Train the model for the current state
    try:
        state_model_pipeline.fit(X_train_state, y_train_state)
    except Exception as e: # Catch potential errors during model fitting
        print(f"  Skipping {state}: Error during model training ({e}).")
        continue

    # Make predictions using the state-specific model
    y_pred_state = state_model_pipeline.predict(X_test_state)

    # Evaluate the model's performance for the current state
    mse_state = mean_squared_error(y_test_state, y_pred_state)
    r2_state = r2_score(y_test_state, y_pred_state)

    print(f"  Mean Squared Error (MSE): {mse_state:.2f}")
    print(f"  R-squared (R2): {r2_state:.2f}")

    # Store the trained model and its evaluation metrics
    state_models[state] = state_model_pipeline
    state_metrics[state] = {'mse': mse_state, 'r2': r2_state}

print("\n--- Summary of State-Wise Model Performance ---")
if not state_metrics:
    print("No models were successfully trained for any state due to insufficient data or errors.")
else:
    # Sort states by R-squared for better readability (highest R2 first)
    sorted_states_metrics = sorted(state_metrics.items(), key=lambda item: item[1]['r2'], reverse=True)
    for state, metrics in sorted_states_metrics:
        print(f"State: {state:<20}, MSE: {metrics['mse']:.2f}, R-squared: {metrics['r2']:.2f}")

print("\nFinished training state-wise models.")


Starting to train models for 30 unique states...

--- Training model for Assam ---
  Mean Squared Error (MSE): 29246.29
  R-squared (R2): 0.99

--- Training model for Karnataka ---
  Mean Squared Error (MSE): 8420.25
  R-squared (R2): 0.99

--- Training model for Kerala ---
  Mean Squared Error (MSE): 22875.42
  R-squared (R2): 0.99

--- Training model for Meghalaya ---
  Mean Squared Error (MSE): 0.16
  R-squared (R2): 0.98

--- Training model for West Bengal ---
  Mean Squared Error (MSE): 5249.74
  R-squared (R2): 1.00

--- Training model for Puducherry ---
  Mean Squared Error (MSE): 25965.31
  R-squared (R2): 0.96

--- Training model for Goa ---
  Mean Squared Error (MSE): 294.83
  R-squared (R2): 1.00

--- Training model for Andhra Pradesh ---
  Mean Squared Error (MSE): 73312.60
  R-squared (R2): 0.96

--- Training model for Tamil Nadu ---
  Mean Squared Error (MSE): 26821.03
  R-squared (R2): 0.99

--- Training model for Odisha ---
  Mean Squared Error (MSE): 0.49
  R-squared (

In [7]:
import os 
import pickle
model_save_dir = 'state_crop_yield_models'
os.makedirs(model_save_dir, exist_ok=True)
print(f"Saving models to directory: {model_save_dir}/")

for state, model_pipeline in state_models.items():
    # Sanitize state name for use in filename (replace spaces and special chars)
    filename = os.path.join(model_save_dir, f'{state.replace(" ", "_").lower()}_crop_yield_model.pkl')
    try:
        with open(filename, 'wb') as f:
            pickle.dump(model_pipeline, f)
        print(f"  Saved model for {state} to {filename}")
    except Exception as e:
        print(f"  Error saving model for {state}: {e}")

print("\nAll state-wise models saved.")

Saving models to directory: state_crop_yield_models/
  Saved model for Assam to state_crop_yield_models\assam_crop_yield_model.pkl
  Saved model for Karnataka to state_crop_yield_models\karnataka_crop_yield_model.pkl
  Saved model for Kerala to state_crop_yield_models\kerala_crop_yield_model.pkl
  Saved model for Meghalaya to state_crop_yield_models\meghalaya_crop_yield_model.pkl
  Saved model for West Bengal to state_crop_yield_models\west_bengal_crop_yield_model.pkl
  Saved model for Puducherry to state_crop_yield_models\puducherry_crop_yield_model.pkl
  Saved model for Goa to state_crop_yield_models\goa_crop_yield_model.pkl
  Saved model for Andhra Pradesh to state_crop_yield_models\andhra_pradesh_crop_yield_model.pkl
  Saved model for Tamil Nadu to state_crop_yield_models\tamil_nadu_crop_yield_model.pkl
  Saved model for Odisha to state_crop_yield_models\odisha_crop_yield_model.pkl
  Saved model for Bihar to state_crop_yield_models\bihar_crop_yield_model.pkl
  Saved model for Gujar